In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from sklearn.metrics import precision_score, recall_score, f1_score

def read_and_merge_csv_files(folder_path):
    # List to hold individual DataFrames
    dataframes = []

    # Loop through all files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            # Read the CSV file into a DataFrame
            df = pd.read_csv(file_path, index_col=0)
            # Append the DataFrame to the list
            dataframes.append(df)

    # Concatenate all DataFrames in the list into a single DataFrame
    merged_df = pd.concat(dataframes, ignore_index=True)

    return merged_df
# Function to split the name column and create new columns
def split_name_column(name):
    parts = name.split('_')
    parameters = parts[-1].replace('.qasm', '').strip('[]')
    position = int(parts[5].replace('P', ''))
    qubit = parts[6].replace('Q', '')
    return parts[0], parts[1], parts[3], parts[4], position, qubit, parameters

def get_gate_type(gate):
    single_qubit_gates = ["x", "h", "p", "t", "s", "z", "y", "id", "rx", "ry", "rz", "sx", "u", "u1", "u2", "u3"]
    multi_qubit_gates = ["swap", "rzz", "rxx", "cx", "cz", "cp", "ccx", "cswap"]
    if gate in single_qubit_gates:
        return 'Single_qubit'
    elif gate in multi_qubit_gates:
        return 'Multi_qubit'
    else:
        return 'Gate_not_supported'
    
# Function to categorize position based on percentage
def categorize_position(percentage):
    if percentage <= 20:
        return 'beginning'
    elif percentage <= 40:
        return 'pre_middle'
    elif percentage <= 60:
        return 'middle'
    elif percentage <= 80:
        return 'post_middle'
    else:
        return 'end'
    


In [ ]:
def getDataframeThreshold(threshold):
    # Get all the results in a df
    folder_path = f'./results/results_{threshold}'
    df = read_and_merge_csv_files(folder_path)
    
    # Create a column to categorize the input type
    df[['Input_type']] = df['Input'].apply(lambda x: pd.Series(x.split('_')[0]))
    
    # Apply the function to the name column and create new columns
    df[['Algorithm', 'Qubits_number',  'Operator', 'Gate', 'Position', 'Qubits', 'Params']] = df['Name'].apply(lambda x: pd.Series(split_name_column(x)))
    
    # Create a column to categorize the gate type
    df[['Gate_type']] = df['Gate'].apply(lambda x: pd.Series(get_gate_type(x)))
    
    # Calculate position percentage and categorize it
    df['max_position'] = df.groupby(['Algorithm', 'Qubits_number'])['Position'].transform('max')
    df['position_percentage'] = (df['Position'] / df['max_position']) * 100
    df['Relative_position'] = df['position_percentage'].apply(categorize_position)
    
    # Drop the intermediate columns if needed
    df = df.drop(columns=['max_position', 'position_percentage'])
    df = df.drop(columns=['Name'])

    return df

# F1, Precission and recall

In [ ]:
metrics = ['C','H','J','T','F','E']
thresholds = ['0.01','0.05','0.1','0']

for threshold in thresholds:
    df = getDataframeThreshold(threshold)
    results = []
    for metric in metrics:
        true_labels = df[f'Killed_I{metric}']
        predicted_labels = df[f'Killed_N{metric}']
        
        # Calculate precision, recall, and F1 score for the pair
        precision = precision_score(true_labels, predicted_labels, zero_division=0)
        recall = recall_score(true_labels, predicted_labels, zero_division=0)
        f1 = f1_score(true_labels, predicted_labels, zero_division=0)
        
        # Append the results as a tuple to the list
        results.append((precision, recall, f1))
    
    # Convert the results into a DataFrame for easy visualization
    results_df = pd.DataFrame(results, columns=['Precision', 'Recall', 'F1 Score'], 
                              index=[f'Metric {metric}' for metric in metrics])
    print(f'Results for threshold {threshold}:')
    print(results_df)

In [ ]:
metrics = ['C','H','J','T','F'] #,'E']
thresholds = ['0.01','0.05','0.1','0']

for threshold in thresholds:
    df = getDataframeThreshold(threshold)
    results = []
    for metric in metrics:
        
        true_rows = df[df[f'Killed_I{metric}'] == False]  # All 'True' rows
        false_rows = df[df[f'Killed_I{metric}'] == True].sample(n=len(true_rows), random_state=42)  # Random sample of 'False' rows, same size as 'True'
        # Concatenate both subsets
        balanced_sample = pd.concat([true_rows, false_rows])
        
        # Shuffle the resulting DataFrame (optional, for randomness in final sample)
        balanced_sample = balanced_sample.sample(frac=1, random_state=42).reset_index(drop=True)
        
        true_labels = balanced_sample[f'Killed_I{metric}']
        predicted_labels = balanced_sample[f'Killed_N{metric}']
        
        # Calculate precision, recall, and F1 score for the pair
        precision = precision_score(true_labels, predicted_labels, zero_division=0)
        recall = recall_score(true_labels, predicted_labels, zero_division=0)
        f1 = f1_score(true_labels, predicted_labels, zero_division=0)
        
        # Append the results as a tuple to the list
        results.append((precision, recall, f1))
    
    # Convert the results into a DataFrame for easy visualization
    results_df = pd.DataFrame(results, columns=['Precision', 'Recall', 'F1 Score'], 
                              index=[f'Metric {metric}' for metric in metrics])
    print(f'Results for threshold {threshold}:')
    print(results_df)

# Confusion matrices


In [ ]:
def confusionMatrix(df, col1, col2):
    # Create a confusion matrix DataFrame
    conf_matrix = pd.DataFrame(index=['True', 'False'], columns=['True', 'False'])
    
    # Calculate the count of each pair
    true_true = ((df[col1] == True) & (df[col2] == True)).sum()
    false_false = ((df[col1] == False) & (df[col2] == False)).sum()
    true_false = ((df[col1] == True) & (df[col2] == False)).sum()
    false_true = ((df[col1] == False) & (df[col2] == True)).sum()
    
    # Total number of rows
    total = len(df)
    
    # Calculate percentages
    conf_matrix.loc['True', 'True'] = (true_true / total) * 100
    conf_matrix.loc['False', 'False'] = (false_false / total) * 100
    conf_matrix.loc['True', 'False'] = (true_false / total) * 100
    conf_matrix.loc['False', 'True'] = (false_true / total) * 100
    
    # Ensure all values are numeric and handle any potential issues
    conf_matrix = conf_matrix.apply(pd.to_numeric, errors='coerce')  # Convert to numeric, coerce errors to NaN
    conf_matrix.fillna(0, inplace=True)  # Replace NaNs with 0 if there are any

    return conf_matrix

In [ ]:
# Define a function to create a heatmap with annotations
def create_heatmap(fig, data, row, col, showscale):
    fig.add_trace(
        go.Heatmap(
            z=data,
            text=data,  # Use the same data for annotations
            colorscale= [[0.0, '#eff3ff'], [0.05, '#9ecae1'],[0.1, '#6baed6'], [0.8, '#3182bd'], [1, '#08519c']],
            colorbar=dict(title='Scale'),
            zmin=0, zmax=100,
            showscale=showscale,
            texttemplate='%{text:.2f}',  # Format the text annotations
            textfont=dict(size=18)
        ),
        row=row, col=col
    )
    
    fig.update_xaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    fig.update_yaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    

In [ ]:
def printConfusionMatrixes(name):
    
    metrics = {'C':'0.01','H':'0.01','J':'0.01','T':'0.01','F':'0.01','E':'0.01'}
    
    fig = make_subplots(rows=1, cols=6,
                            subplot_titles=('Chisquare', 'Hellinger', 'Jensen-shannon', 'Trace', 'Fidelity', 'Expectation Values'), x_title='Noisy', y_title='Ideal', horizontal_spacing=0.05)
    
    for j,metric in enumerate(metrics.keys()):
        threshold = metrics[metric]
        df_confusion = getDataframeThreshold(threshold)
        confusion_matrix = confusionMatrix(df_confusion, 'Killed_I' + metric,'Killed_N' + metric)
        # Add heatmaps to subplots
        create_heatmap(fig, confusion_matrix, row=1, col=j+1, showscale=True)
    
    fig.update_layout(
        title_text=name,
        height=400,
        width=2000,
        showlegend=False
    )
    
    fig.show()
    # fig.write_image("images/" + name + ".png", engine='orca')


In [ ]:
# OVERALL CONFUSION MATRIX
printConfusionMatrixes('Overall confusion matrix')

# Load program characteristics

In [ ]:
 # Load the .xlsx file
file_path = 'data/origin_qc/programs_characteristics.xlsx'
df_charac = pd.read_excel(file_path, usecols=[0, 2, 3, 5, 6, 7])
df_charac['algo'] = df_charac.iloc[:, 0].str.split('_').str[0]  # Extract the algorithm name (first part)
df_charac['qubits'] = df_charac['qubits'].astype(str)
df_charac = df_charac.drop(columns=[df_charac.columns[0]])

# Merge df and df_charac on 'qubits'/'Qubits' and 'algo'/'Algorithm'
merged_df = pd.merge(df_charac, df, left_on=['qubits', 'algo'], right_on=['Qubits_number', 'Algorithm'], how='right')
merged_df = merged_df.drop(columns=['qubits'])  # or 'Qubits_number' if you prefer to keep the original name
df = merged_df.drop(columns=['algo'])  # or 'Qubits_number' if you prefer to keep the original name

print(df.columns)

# "Output_type": ["Dominent", "Diverse"],

In [ ]:
metrics = {'C':'0.01','H':'0.01','J':'0.01','T':'0.01','F':'0.01','E':'0.01'}
metric_names=('Chisquare', 'Hellinger', 'Jensen-shannon', 'Trace', 'Fidelity', 'Expectation Values')

# Line Graphs

In [ ]:
def print_line_chart(name, categories, metrics, metric_names):
    
    fig = go.Figure()
    color_scale = px.colors.qualitative.Bold
    scores = {}
    
    # Loop over the groups to add each one to the figure
    for i, metric in enumerate(metrics.keys()):
        
        threshold = metrics[metric]
        df_threshold = getDataframeThreshold(threshold)   
        need_interpolation = False
        
        for cat in categories:
            
            if name == 'Qubits_number':
                filtered_df = df_threshold.loc[df[name] == cat]
            else:
                filtered_df = df_threshold.loc[df[name] == int(cat)]
            
            if filtered_df.empty:
                scores[cat, metric] = None  # Or you can assign NaN or a default value
                need_interpolation = True
            else:
                true_labels = filtered_df[f'Killed_I{metric}']
                predicted_labels = filtered_df[f'Killed_N{metric}']
                # Calculate F1 score for the pair
                scores[cat, metric] = f1_score(true_labels, predicted_labels, zero_division=0)
    
    
        if need_interpolation:
            # Assuming scores is a dictionary structured as {(category, metric): score}
            scores_df = pd.DataFrame.from_dict(scores, orient='index', columns=['F1_Score'])
            scores_df.reset_index(inplace=True)
            scores_df.columns = ['Category_Metric', 'F1_Score']
            
            # Use interpolation to fill missing values
            scores_df['F1_Score'] = scores_df['F1_Score'].interpolate(method='linear')
            
            # Convert back to a dictionary if needed
            scores_interpolated = {(row['Category_Metric']): row['F1_Score'] for _, row in scores_df.iterrows()}
        
            y_values = [scores_interpolated[(cat, metric)] for cat in categories]
            
        else:
            y_values = [scores[(cat, metric)] for cat in categories]
            
        
        fig.add_trace(go.Scatter(
            name=metric_names[i],  # Name of the operator
            x=categories,  # Metrics on x-axis
            y=y_values,  # F1 scores for the current operator
            hoverinfo='y',  # Show F1 score on hover
            marker=dict(color=color_scale[i % len(color_scale)])  # Assign a color from the color scale
        ))
        
    fig.update_layout(
        title_text=name,
        height=400,
        width=2000,
        showlegend=True
    )
    
    fig.show()
    #fig.write_image("images/" + name + ".png", engine='orca')

In [ ]:
columns = ['Qubits_number', 'gates', 'depth', 'singlequbit_gates', 'multiqubit_gates'] 

for cat in columns: 
    min_val = int(df[cat].min())
    max_val = int(df[cat].max())
    cat_range = list(map(str, range(min_val, max_val + 1)))  
    print_line_chart(cat, cat_range, metrics, metric_names)

# Bar graphs

In [ ]:
def print_grouped_bar_chart(name, categories, metrics, metric_names):
    
    scores = {}
    
    # Loop over the groups to add each one to the figure
    for i, metric in enumerate(metrics.keys()):
        
        threshold = metrics[metric]
        df_threshold = getDataframeThreshold(threshold)
        
        for cat in categories:
            
            filtered_df = df_threshold.loc[df[name] == cat]
            
            true_labels = filtered_df[f'Killed_I{metric}']
            predicted_labels = filtered_df[f'Killed_N{metric}']
            
            # Calculate F1 score for the pair
            scores[cat, metric] = f1_score(true_labels, predicted_labels, zero_division=0)
                
    # Create the bar graph
    fig = go.Figure()
    color_scale = px.colors.qualitative.Bold
    
    # Add bars for each category
    for i, cat in enumerate(categories):
        # Extract scores for the current category across all metrics
        y_values = [scores[(cat, metric)] for metric in metrics]
        fig.add_trace(go.Bar(
            name=cat,  # Name of the metric
            x=metric_names,  # Metrics on x-axis
            y=y_values,  # F1 scores for the current metric
            hoverinfo='y',  # Show F1 score on hover
            marker=dict(color=color_scale[i % len(color_scale)])  # Assign a color from the color scale
        ))
        
    fig.update_layout(
        title_text=name,
        height=400,
        width=2000,
        showlegend=True
    )
    
    fig.show()
    #fig.write_image("images/" + name + ".png", engine='orca')

In [ ]:
table_data = {
    # "Output_type": ["Dominent", "Diverse"],
    "Input_type": ["PureState", "Quratest"],
    "Operator": ["Add", "Remove", "Replace"],
    "Gate_type": ["Single_qubit", "Multi_qubit"],
    "Relative_position": ["beginning", "pre_middle", "middle", "post_middle", "end"]
}

for key, categories in table_data.items():
    print_grouped_bar_chart(key, categories, metrics, metric_names)